In [3]:
import xarray as xr

import numpy as np

import scipy.stats as stats

import pandas as pd

from sklearn.cluster import KMeans

from sklearn.preprocessing import StandardScaler

import glob

import datetime

import matplotlib.pyplot as plt

import pickle

import cartopy.crs as ccrs

import cartopy.feature as cfeature

from scipy.stats import spearmanr

import seaborn as sns

import os

import matplotlib.colors as mcolors

from matplotlib.colors import TwoSlopeNorm

from sklearn.metrics import silhouette_score

from scipy.stats import pearsonr

from pathlib import Path

from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

import metpy.calc as mpcalc

from metpy.constants import Rd

from metpy.units import units

pd.options.mode.chained_assignment = None

In [2]:
import numpy as np

data1 = np.load(
    "/home/annierosen16/backup_data/backup_arrays_for_buoyancy_12lst_1979_2000.npz",
    mmap_mode="r"
)

data2 = np.load(
    "/home/annierosen16/backup_data/backup_arrays_for_buoyancy_12lst_2001_2025.npz",
    mmap_mode="r"
)

# These DO NOT load full arrays into RAM
p1_shape = data1["array1"].shape
p2_shape = data2["array1"].shape

print("p1 shape:", p1_shape)
print("p2 shape:", p2_shape)

# Combined pressure shape (without concatenating)
combined_shape = (
    p1_shape[0] + p2_shape[0],  # time dimension adds
    p1_shape[1],
    p1_shape[2],
    p1_shape[3],
)

print("Combined pressure shape:", combined_shape)

p1 shape: (3366, 201, 37, 41)
p2 shape: (3825, 201, 37, 41)
Combined pressure shape: (7191, 201, 37, 41)


## preparing arrays for era5_compute_ctp_hilow.py script

In [ ]:
data1 = np.load("/home/annierosen16/backup_data/backup_arrays_for_buoyancy_12lst_1979_2000.npz")

data2 = np.load("/home/annierosen16/backup_data/backup_arrays_for_buoyancy_12lst_2001_2025.npz")

# Pull arrays (same names in both)

p1 = data1["array1"]   # (time, z, lat, lon)

print("p1 loaded")

q1 = data1["array2"]

print("q1 loaded")

t1 = data1["array3"]

print("t1 loaded")

p2 = data2["array1"]

print("p2 loaded")

q2 = data2["array2"]

print("q2 loaded")

t2 = data2["array3"]

print("t2 loaded")

# Concatenate along time

pressure = np.concatenate([p1, p2], axis=0)

print("pressure combined")

specific_humidity = np.concatenate([q1, q2], axis=0)

print("specific humidity combined")

temperature = np.concatenate([t1, t2], axis=0) + 273.15  # convert C -> K after concat (or before, same result)

print("temperature combined")

print("pressure:", pressure.shape)

print("q:", specific_humidity.shape)

print("T (K):", temperature.shape)

# Altitude grid (0–4000 m every 20 m)

altitude = np.arange(0, 4000 + 20, 20)

print("altitude:", altitude.shape)

p1 loaded
q1 loaded
t1 loaded
p2 loaded
q2 loaded


In [ ]:
# # these are all interpolated to every 20 m

# pressure = data['array1'] # Shape: (time, pressure_level, lat, lon)

# print("p loaded")

# specific_humidity = data['array2'] # Shape: (time, pressure_level, lat, lon)

# print("q loaded")

# temperature = data['array3'] # Shape: (time, pressure_level, lat, lon) in Celsius

# temperature = temperature + 273.15 # converting to Kelvin

# print("t loaded")

# altitude = np.arange(0, 4000 + 20, 20)

# # calculating dewpoint from specific humidity 

# # temp conversion to kelvin necessary for dewpoint calculation 

# temperature = (temperature + 273.15) * units.K

# print("t complete")

# pressure = pressure * units.hPa  

# print("p complete")

# humidity = q * units('g/kg')

# print("q complete")

# dewpoint = mpcalc.dewpoint_from_specific_humidity(pressure, temperature, humidity)

# print("dewpoint complete")

In [3]:
nt, nz, ny, nx = pressure.shape

# Use float32 to reduce memory

p = pressure.astype(np.float32)      # hPa

print("p converted to float32")

T = temperature.astype(np.float32)   # K

print("T converted to float32")

q = specific_humidity.astype(np.float32) # g/kg

print("q converted to float32")

# Output dewpoint in Celsius

td_out = np.memmap(
    
    "/home/annierosen16/backup_data/dewpoint_for_ctp_hilow_1979_2025.dat",
    
    dtype="float32",
    
    mode="w+",
    
    shape=(nt, nz, ny, nx)
)

print("td_out created")

chunk = 10  # adjust upward if stable

for t0 in range(0, nt, chunk):
    
    t1 = min(t0 + chunk, nt)
    
    print(f"MetPy dewpoint: {t0}:{t1}")

    # Attach units ONLY to the chunk
    
    p_chunk = p[t0:t1] * units.hPa
    
    T_chunk = T[t0:t1] * units.K
    
    q_chunk = q[t0:t1] * units("g/kg")

    # MetPy returns dewpoint in temperature units (°C by default)
    
    td_chunk = mpcalc.dewpoint_from_specific_humidity(
        
        p_chunk, T_chunk, q_chunk)

    # Store magnitudes in °C
    
    td_out[t0:t1] = td_chunk.magnitude.astype(np.float32)

td_out.flush()

print("done")

p converted to float32
T converted to float32
q converted to float32
td_out created
MetPy dewpoint: 0:10
MetPy dewpoint: 10:20
MetPy dewpoint: 20:30
MetPy dewpoint: 30:40
MetPy dewpoint: 40:50
MetPy dewpoint: 50:60
MetPy dewpoint: 60:70
MetPy dewpoint: 70:80
MetPy dewpoint: 80:90
MetPy dewpoint: 90:100
MetPy dewpoint: 100:110
MetPy dewpoint: 110:120
MetPy dewpoint: 120:130
MetPy dewpoint: 130:140
MetPy dewpoint: 140:150
MetPy dewpoint: 150:160
MetPy dewpoint: 160:170
MetPy dewpoint: 170:180
MetPy dewpoint: 180:190
MetPy dewpoint: 190:200
MetPy dewpoint: 200:210
MetPy dewpoint: 210:220
MetPy dewpoint: 220:230
MetPy dewpoint: 230:240
MetPy dewpoint: 240:250
MetPy dewpoint: 250:260
MetPy dewpoint: 260:270
MetPy dewpoint: 270:280
MetPy dewpoint: 280:290
MetPy dewpoint: 290:300
MetPy dewpoint: 300:310
MetPy dewpoint: 310:320
MetPy dewpoint: 320:330
MetPy dewpoint: 330:340
MetPy dewpoint: 340:350
MetPy dewpoint: 350:360
MetPy dewpoint: 360:370
MetPy dewpoint: 370:380
MetPy dewpoint: 380:390


MetPy dewpoint: 3230:3240
MetPy dewpoint: 3240:3250
MetPy dewpoint: 3250:3260
MetPy dewpoint: 3260:3270
MetPy dewpoint: 3270:3280
MetPy dewpoint: 3280:3290
MetPy dewpoint: 3290:3300
MetPy dewpoint: 3300:3310
MetPy dewpoint: 3310:3320
MetPy dewpoint: 3320:3330
MetPy dewpoint: 3330:3340
MetPy dewpoint: 3340:3350
MetPy dewpoint: 3350:3360
MetPy dewpoint: 3360:3370
MetPy dewpoint: 3370:3380
MetPy dewpoint: 3380:3390
MetPy dewpoint: 3390:3400
MetPy dewpoint: 3400:3410
MetPy dewpoint: 3410:3420
MetPy dewpoint: 3420:3430
MetPy dewpoint: 3430:3440
MetPy dewpoint: 3440:3450
MetPy dewpoint: 3450:3460
MetPy dewpoint: 3460:3470
MetPy dewpoint: 3470:3480
MetPy dewpoint: 3480:3490
MetPy dewpoint: 3490:3500
MetPy dewpoint: 3500:3510
MetPy dewpoint: 3510:3520
MetPy dewpoint: 3520:3530
MetPy dewpoint: 3530:3540
MetPy dewpoint: 3540:3550
MetPy dewpoint: 3550:3560
MetPy dewpoint: 3560:3570
MetPy dewpoint: 3570:3580
MetPy dewpoint: 3580:3590
MetPy dewpoint: 3590:3600
MetPy dewpoint: 3600:3610
MetPy dewpoi

MetPy dewpoint: 6390:6400
MetPy dewpoint: 6400:6410
MetPy dewpoint: 6410:6420
MetPy dewpoint: 6420:6430
MetPy dewpoint: 6430:6440
MetPy dewpoint: 6440:6450
MetPy dewpoint: 6450:6460
MetPy dewpoint: 6460:6470
MetPy dewpoint: 6470:6480
MetPy dewpoint: 6480:6490
MetPy dewpoint: 6490:6500
MetPy dewpoint: 6500:6510
MetPy dewpoint: 6510:6520
MetPy dewpoint: 6520:6530
MetPy dewpoint: 6530:6540
MetPy dewpoint: 6540:6550
MetPy dewpoint: 6550:6560
MetPy dewpoint: 6560:6570
MetPy dewpoint: 6570:6580
MetPy dewpoint: 6580:6590
MetPy dewpoint: 6590:6600
MetPy dewpoint: 6600:6610
MetPy dewpoint: 6610:6620
MetPy dewpoint: 6620:6630
MetPy dewpoint: 6630:6640
MetPy dewpoint: 6640:6650
MetPy dewpoint: 6650:6660
MetPy dewpoint: 6660:6670
MetPy dewpoint: 6670:6680
MetPy dewpoint: 6680:6690
MetPy dewpoint: 6690:6700
MetPy dewpoint: 6700:6710
MetPy dewpoint: 6710:6720
MetPy dewpoint: 6720:6730
MetPy dewpoint: 6730:6740
MetPy dewpoint: 6740:6750
MetPy dewpoint: 6750:6760
MetPy dewpoint: 6760:6770
MetPy dewpoi

In [10]:
# # -------------------------------------------------
# # 1) Load pressure, temperature, humidity from NPZ
# # -------------------------------------------------
# npz_path = "/home/annierosen16/backup_data/backup_arrays_for_buoyancy_12lst_2001_2025.npz"

# data = np.load(npz_path)

# print("npz file loaded")

# # Arrays interpolated every 20 m AGL

# pressure = data["array1"]        # hPa, shape (time, level, lat, lon)

# # Altitude grid (AGL)
# altitude = np.arange(0, 4000 + 20, 20) * units.meter

# print("units attached")

# -------------------------------------------------
# 2) Load dewpoint from memmap (.dat)
# -------------------------------------------------

dat_path = "/home/annierosen16/backup_data/dewpoint_for_ctp_hilow_1979_2025.dat"

# Shape MUST match pressure

td = np.memmap(
    
    dat_path,
    
    dtype="float32",
    
    mode="r",
    
    shape=combined_shape
)

# Attach units (°C)

dewpoint79 = td * units.degC

print("dewpoint loaded")

print("Td range (°C):", np.nanmin(dewpoint79.magnitude), np.nanmax(dewpoint79.magnitude))

dewpoint loaded
Td range (°C): -106.492676 28.398773


In [11]:
dat_path = "/home/annierosen16/backup_data/dewpoint_for_ctp_hilow_2001_2025.dat"

# Shape MUST match pressure

td = np.memmap(
    
    dat_path,
    
    dtype="float32",
    
    mode="r",
    
    shape=p2_shape
)

# Attach units (°C)

dewpoint01 = td * units.degC

print("dewpoint loaded")

print("Td range (°C):", np.nanmin(dewpoint01.magnitude), np.nanmax(dewpoint01.magnitude))

dewpoint loaded
Td range (°C): -106.492676 28.398773


In [7]:
# # temp conversion back to deg C necessary for cal_ctp_hi 

# temperature = temperature.magnitude

# temperature = (temperature - 273.15) * units.degC

In [9]:
# # Remove the units (store only the raw values)

# temperature_data = temperature.magnitude

# dewpoint_data = dewpoint.magnitude

# pressure_data = pressure.magnitude

# altitude_data = altitude  

# # Create the directory if it doesn't exist

# backup_data_dir = os.path.expanduser('~/backup_data')

# os.makedirs(backup_data_dir, exist_ok=True)

# # Define the file path

# file_path = os.path.join(backup_data_dir, 'ctp_hilow_input_arrays_2001_2025.npz')

# # Save the arrays into the .npz file

# np.savez(file_path, temperature=temperature_data, dewpoint=dewpoint_data, pressure=pressure_data, altitude=altitude_data)

# print(f"Data saved successfully to {file_path}")

In [ ]:
# # Define the file paths where the data was saved

# ctp_file_path = '/data/rong3/annie/completed_ctp_hilow_arrays/ctp_results.npz'

# hi_file_path = '/data/rong3/annie/completed_ctp_hilow_arrays/hi_results.npz'

# # Load the arrays from the .npz file
# ctp_data = np.load(ctp_file_path)

# hi_data = np.load(hi_file_path)

# # Extract the arrays from the loaded data
# ctp_results = ctp_data['ctp_results']

# hi_results = hi_data['hi_results']

# # contains lists of CTP for every month-day

# ctpdf = resultdf.groupby(['month', 'day'])['ctp'].agg(list).reset_index()

# ctpdf['ctpz']= ""

# for i in range(ctpdf.shape[0]):
    
#     # extracts all CTPs for the month-day 
    
#     x = np.array(ctpdf['ctp'][i])

#     x = x.reshape(-1, 1)

#     qt = QuantileTransformer(output_distribution='normal')
    
#     xqt = qt.fit_transform(x)
    
#     ctpdf['ctpz'][i] = xqt
    
# #output of xqt puts every value in a list, this removes the inner lists

# ctpdf['ctpz'] = ctpdf['ctpz'].apply(lambda x: [item for sublist in x for item in sublist])  

# ctp_raw = ctpdf.explode('ctp').drop('ctpz', axis=1)

# ctp_z = ctpdf.explode('ctpz').drop('ctp', axis=1)

# ctpz = ctp_z['ctpz']

# ctpdf = pd.concat([ctp_raw, ctpz], axis=1)

# resultdf = resultdf.reset_index()

# ctpdf = ctpdf.reset_index()

# resultdf = pd.merge(resultdf, ctpdf[['month','day', 'ctp', 'ctpz']], on=['month', 'day', 'ctp'], how='left')

In [10]:
# def cal_ctp_hi(temperature, dewpoint, pressure, altitude):
    
#     '''
#     INPUT:
#         temperature 
#         dewpoint
#         pressure
#         altitude 
#     Note: all input should have metpy.units but altitude
#     OUTPUT:
#         ctp  : Convective Triggering Potential
#         hi   : Humidity Index
#         moist_t : Moist adiabatic temperature profile from 1km to 3km AGL
#         cal_p   : Pressure profile from 1km to 3km AGL
#     '''
    
#     alt = altitude 
    
#     start_index = np.where(alt == alt[0] + 1000)[0][0]
    
#     end_index = np.where(alt == alt[0] + 3000)[0][0]
    
#     cal_p = pressure[start_index:end_index + 1]
        
#     cal_t = temperature[start_index:end_index + 1]
    
#     # to interpolate observed pressure profile to 2 hpa increments
    
#     p_level = np.arange(pressure[0].magnitude, pressure[-1].magnitude, -2) * units.hPa
    
#     # calculates a moist temperature profile against synthetic pressure levels (p_level)
    
#     # and using starting temperature and pressure as starting point
    
#     moist_temp = mpcalc.moist_lapse(
        
#         pressure=p_level, 
        
#         temperature=cal_t[0], 
        
#         reference_pressure=cal_p[0]
    
#     ).to(units.degC)
    
#     # cal_moist is segmented from moist_temp to 100 -> 300 hpa AGL
    
#     cal_moist = np.interp(
        
#         cal_p.magnitude[::-1],
        
#         p_level.magnitude[::-1],
        
#         moist_temp.magnitude[::-1]
    
#     )[::-1] * moist_temp.units
        
#     ctp = (Rd * (np.trapz(
        
#         (cal_moist.magnitude - cal_t.magnitude)[::-1], 
        
#         np.log(cal_p.magnitude)[::-1]
    
#     ) * units.degK)).to(units('J/kg'))
    
#     index0 = np.where(alt == alt[0] + 500)[0][0]
    
#     index1 = np.where(alt == alt[0] + 1500)[0][0]
    
#     hi = (temperature[index0] - dewpoint[index0] + 
          
#           temperature[index1] - dewpoint[index1])
    
#     print(temperature[index0] - dewpoint[index0],temperature[index1] - dewpoint[index1])
    
#     return ctp.magnitude, hi.magnitude

In [1]:
# print(ctp, "J/kg")

# plt.plot(cal_moist, cal_p, label='moist adiabat')

# plt.plot(cal_t, cal_p, color='red', label='observed t')

# plt.scatter(moist_temp, p_level, color='orange', s=1)

# plt.gca().invert_yaxis()

# plt.legend()

# plt.grid()

# plt.show()

In [2]:
# print(hi)

# plt.plot(temperature_test, altitude, label='temp')

# plt.plot(dewpoint_test, altitude, label='dp')

# plt.legend()

# plt.grid()

# plt.show()